# Spark-X2.6-4B-En4B-A1B (`v0.3`) — 72-Layer 16K Unsloth + Triton Full-Finetuning Notebook

This notebook runs the unattended **16K (`16,384` tokens)** full-finetuning and **262K (`262,144` tokens)** multi-turn agentic verification pipeline for **`Spark-X2.6-4B-En4B-A1B` (`v0.3`)**.

All core architecture modules ([`sparkx_loop_v03_core.py`](./sparkx_loop_v03_core.py)), training & dataset curation pipelines ([`sparkx_loop_v03_train_env.py`](./sparkx_loop_v03_train_env.py)), and standalone inference loaders ([`load_inference_v03.py`](./load_inference_v03.py)) live directly in the GitHub repository (`BaiTian6641/Spark-X2.6-Recursion`).

### Architecture & Kernel Highlights
- **Backbone**: Clean 1M-context `XHToken/Spark-X2.6-4B` (`4.23B` dense parameters, 36 physical layers).
- **3-Pass Telescoping Recurrence (`72` Virtual Layers)**:
  - Prelude `[0..4)` (4L) + Pass 1 `[4..28)` (24L) + Pass 2 `[4..28)` (24L) + Pass 3 `[12..24)` (12L) + Coda `[28..36)` (8L) = **72 Effective Layers**.
  - Exact Zero-Gated Step-0 Identity (`step0_diff == 0.0` between 36L 1-pass and 72L 3-pass at initialization).
- **8-Anchor 3-Tier Zipfian MoE-Engram (`4.11B` Total / `1.10B` Active)**:
  - Placed at physical layers `3, 7, 11, 15, 19, 23, 27, 31` (`8 -> 32` coarse-to-fine expert upcycling at Stage 3).
- **16K Unsloth + Triton + Liger + Cut-Cross-Entropy Kernel Stack**:
  - Custom `@triton.jit` Fused SwiGLU & RMSNorm kernels (`triton_swiglu`, `triton_rmsnorm`), zero-sync sorted MoE dispatch, and gradient-checkpointed `fused_masked_cross_entropy` over the 167,168-token vocabulary.

In [ ]:
# Cell 1: Clone / Sync Repository & Install 16K Unsloth + Triton + Cut-CE + Liger Stack
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = Path('/content/Spark-X2.6-Recursion')

if (Path.cwd() / '.git').exists() and (Path.cwd() / 'sparkx_loop_v03_core.py').exists():
    REPO_DIR = Path.cwd()
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
elif REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=False)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

pip_pkgs = [
    'transformers==4.57.1',
    'accelerate>=1.2.0',
    'datasets>=3.2.0',
    'safetensors>=0.4.5',
    'huggingface_hub>=0.27.0',
    'bitsandbytes>=0.45.0',
    'triton>=3.0.0',
    'liger-kernel',
    'cut-cross-entropy',
    'unsloth_zoo',
    'unsloth',
    'openai>=1.58.0',
    'psutil',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pip_pkgs], check=False)
print('Working directory synced:', Path.cwd())

In [ ]:
# Cell 2: Load Training Environment Config, Secrets & 16K Hardware Auto-Scaler
import importlib, os
import sparkx_loop_v03_core
import sparkx_loop_v03_train_env
importlib.reload(sparkx_loop_v03_core)
importlib.reload(sparkx_loop_v03_train_env)

from sparkx_loop_v03_core import (
    LoopV03Settings,
    SparkXLoopV03,
    patch_spark_for_unsloth_triton,
)
from sparkx_loop_v03_train_env import (
    DEFAULT_MAX_NEW_TOKENS,
    SparkXAgenticWorkspace,
    auto_configure_vram_16k,
    build_16k_stage_pools,
    build_default_cfg,
    curate_all_pillars_v03,
    generate_opencode_go_trajectories,
    run_step0_verification_suite,
    run_v03_four_stage_training,
    set_loglinear_kv_runtime,
)

def _get_secret(name: str):
    val = os.environ.get(name)
    if val:
        return val.strip()
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    return None

HF_TOKEN = _get_secret('HF_TOKEN')
OPENCODE_API_KEY = _get_secret('OPENCODE_API_KEY')

CFG = build_default_cfg()
CFG = auto_configure_vram_16k(CFG)

In [ ]:
# Cell 3: Instantiate Spark-X2.6-4B-En4B-A1B (v0.3), Apply Unsloth/Triton Patches & Run Step-0 Audit
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

if 'tokenizer' not in globals() or tokenizer is None:
    print(f"Loading tokenizer from {CFG['BASE_MODEL_ID']}...")
    tokenizer = AutoTokenizer.from_pretrained(CFG['BASE_MODEL_ID'], trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

if 'base_model' not in globals() or base_model is None:
    print(f"Loading clean 1M-context base model from {CFG['BASE_MODEL_ID']}...")
    base_model = AutoModelForCausalLM.from_pretrained(
        CFG['BASE_MODEL_ID'],
        trust_remote_code=True,
        dtype=dtype,
    ).to(device)

patch_info = patch_spark_for_unsloth_triton(base_model)
print('Applied Unsloth/Triton backbone patches:', patch_info)

if 'model' not in globals() or model is None:
    settings = LoopV03Settings(
        start=CFG['LOOP_START'],
        end=CFG['LOOP_END'],
        passes=2,
        p3_start=CFG['P3_START'],
        p3_end=CFG['P3_END'],
        enable_p3=True,
        use_mhc=True,
        use_engram=True,
        checkpoint_layers=True,
        offload_checkpoint_to_cpu=CFG['OFFLOAD_CHECKPOINT_CPU'],
        eng_hot_buckets=CFG['ENG_HOT_BUCKETS'],
        eng_cold_buckets=CFG['ENG_COLD_BUCKETS'],
        eng_dim=CFG['ENG_DIM'],
        eng_shared_rank=CFG['ENG_SHARED_RANK'],
        eng_experts=CFG['ENG_EXPERTS'],
        eng_initial_active_experts=CFG['ENG_INITIAL_ACTIVE_EXPERTS'],
        eng_top_k=CFG['ENG_TOP_K'],
        eng_initial_top_k=CFG['ENG_INITIAL_TOP_K'],
        eng_expert_rank=CFG['ENG_EXPERT_RANK'],
        eng_route_dim=CFG['ENG_ROUTE_DIM'],
        backbone_engram_stride=CFG['ENG_STRIDE'],
        eng_offload_tier1b_cpu=CFG['ENG_OFFLOAD_TIER1B_CPU'],
        eng_z_loss_coeff=CFG['ENG_Z_LOSS_COEFF'],
        ce_chunk_tokens=CFG['CE_CHUNK_TOKENS'],
    )
    model = SparkXLoopV03(base_model, settings=settings).to(device)
    model.controllers.to(device=device)
    model.engram.to(device=device)
    model.backbone_engrams.to(device=device)

# Verify all 5 Step-0 mathematical invariants (including exact 0.0 36L vs 72L parity)
audit_metrics = run_step0_verification_suite(model, base_model, tokenizer)

In [ ]:
# Cell 4: Stream 5-Pillar + Multi-Turn Datasets, Run OpenCode Go Teacher & Pack 16K STAGE_POOLS
curated_records = curate_all_pillars_v03(CFG)
teacher_records = generate_opencode_go_trajectories(CFG, api_key=OPENCODE_API_KEY)
all_records = curated_records + teacher_records

STAGE_POOLS = build_16k_stage_pools(all_records, tokenizer, CFG)

In [ ]:
# Cell 5: Execute 4-Stage Progressive 16K Full-Finetuning (36L -> 48L -> 60L Upcycle32 -> 72L)
train_summary = run_v03_four_stage_training(
    model=model,
    tokenizer=tokenizer,
    stage_pools=STAGE_POOLS,
    cfg=CFG,
    hf_token=HF_TOKEN,
)
print(f"Training complete at global_step={train_summary['global_step']} | Exported to: {train_summary['final_dir']}")

In [ ]:
# Cell 6: Enable O(n log n) Log-Linear KV Cache & Run 262K 72-Layer Agentic Verification
import html
from IPython.display import HTML, display

model.set_loop(
    start=CFG['LOOP_START'],
    end=CFG['LOOP_END'],
    passes=2,
    use_mhc=True,
    use_engram=True,
    enable_p3=True,
    p3_start=CFG['P3_START'],
    p3_end=CFG['P3_END'],
)
set_loglinear_kv_runtime(model, enabled=True, exact_threshold=2048)

workspace = SparkXAgenticWorkspace('/content/sakura_agent_workspace_v03', model=model, tokenizer=tokenizer)
sakura_mission = (
    'Build a complete, self-contained, single-file HTML5 + Three.js 3D Interactive Voxel Cherry Blossom '
    '(Sakura) Garden webpage (`<!DOCTYPE html>` ... `</html>`) and save it as `index.html`.\n'
    'Requirements:\n'
    '1. Load Three.js (`https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js`).\n'
    '2. Use `THREE.InstancedMesh` to render a stepped voxel island terrain with a reflective pond, stepping stones, and red Torii gate / wooden arched bridge.\n'
    '3. Build multiple gnarled dark-wood cherry blossom trees with layered pink/blush voxel canopies and Japanese stone lanterns (`Toro`) with warm point lights.\n'
    '4. Animate at least 400 falling pink sakura petals drifting in the wind and floating on the pond.\n'
    '5. Include interactive mouse-drag orbit & scroll-wheel zoom camera controls, auto-rotate toggle, and HUD buttons for Spring Daylight / Golden Sunset / Lantern Night.'
)

mission_result = workspace.run_agentic_mission(
    sakura_mission,
    target_filename='index.html',
    max_turns=3,
    max_new_tokens_per_turn=DEFAULT_MAX_NEW_TOKENS,
)

if mission_result.get('html'):
    escaped = html.escape(mission_result['html'], quote=True)
    display(HTML(f'<iframe srcdoc="{escaped}" width="100%" height="580" style="border:1px solid #f48fb1; border-radius:10px;"></iframe>'))